# MOUNT GOOGLE DRIVE

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# IMPORTS

In [2]:
import os
import re
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from glob import glob

# PATH CONFIGURATION

In [3]:
BASE_PATH = "/content/drive/MyDrive/Project work/Dataset/lgg-mri-segmentation/lgg_structured"

TRAIN_IMG_DIR  = os.path.join(BASE_PATH, "train/images")
TRAIN_MASK_DIR = os.path.join(BASE_PATH, "train/masks")

VAL_IMG_DIR  = os.path.join(BASE_PATH, "val/images")
VAL_MASK_DIR = os.path.join(BASE_PATH, "val/masks")

TEST_IMG_DIR  = os.path.join(BASE_PATH, "test/images")
TEST_MASK_DIR = os.path.join(BASE_PATH, "test/masks")

MODEL_DIR = "/content/drive/MyDrive/Project work/models/Segmentation"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_KERAS = os.path.join(MODEL_DIR, "brain_segmentation.keras")
MODEL_H5    = os.path.join(MODEL_DIR, "brain_segmentation.h5")

# CONFIGURATION

In [4]:
IMG_SIZE   = 384
BATCH_SIZE = 8
EPOCHS     = 80
WARMUP_EPOCHS = 3
TARGET_DICE = 0.80
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

# DATASET LOADING

In [5]:
def numeric_sort(filepath):
    nums = re.findall(r"\d+", os.path.basename(filepath))
    return [int(n) for n in nums]

train_images = sorted(glob(os.path.join(TRAIN_IMG_DIR, "*")), key=numeric_sort)
train_masks  = sorted(glob(os.path.join(TRAIN_MASK_DIR, "*")), key=numeric_sort)

val_images = sorted(glob(os.path.join(VAL_IMG_DIR, "*")), key=numeric_sort)
val_masks  = sorted(glob(os.path.join(VAL_MASK_DIR, "*")), key=numeric_sort)

test_images = sorted(glob(os.path.join(TEST_IMG_DIR, "*")), key=numeric_sort)
test_masks  = sorted(glob(os.path.join(TEST_MASK_DIR, "*")), key=numeric_sort)

print("Train:", len(train_images))
print("Val:", len(val_images))
print("Test:", len(test_images))

Train: 3133
Val: 409
Test: 387


# DATASET VERIFICATION

In [6]:
print("\nChecking sample pairs:")
for i in range(min(5, len(train_images))):
    print(os.path.basename(train_images[i]), " | ", os.path.basename(train_masks[i]))


Checking sample pairs:
TCGA_FG_A4MT_20020212_1.tif  |  TCGA_FG_A4MT_20020212_1_mask.tif
TCGA_FG_A4MT_20020212_2.tif  |  TCGA_FG_A4MT_20020212_2_mask.tif
TCGA_FG_A4MT_20020212_3.tif  |  TCGA_FG_A4MT_20020212_3_mask.tif
TCGA_FG_A4MT_20020212_4.tif  |  TCGA_FG_A4MT_20020212_4_mask.tif
TCGA_FG_A4MT_20020212_5.tif  |  TCGA_FG_A4MT_20020212_5_mask.tif


# PREPROCESSING

In [7]:
def load_image(path):
    img = cv2.imread(path.numpy().decode(), cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, -1)
    return img


def load_mask(path):
    mask = cv2.imread(path.numpy().decode(), cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 127).astype(np.float32)
    mask = np.expand_dims(mask, -1)
    return mask


def parse_sample(img, mask):
    img  = tf.py_function(load_image, [img], tf.float32)
    mask = tf.py_function(load_mask,  [mask], tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 1])
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return img, mask

# STRONG AUGMENTATION

In [8]:
def augment(img, mask):

    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)

    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)

    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    img  = tf.image.rot90(img, k)
    mask = tf.image.rot90(mask, k)

    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.8, 1.2)

    gamma = tf.random.uniform([], 0.8, 1.2)
    img = tf.image.adjust_gamma(img, gamma)

    noise = tf.random.normal(tf.shape(img), 0, 0.03)
    img = tf.clip_by_value(img + noise, 0, 1)

    mask = tf.cast(mask > 0.5, tf.float32)

    return img, mask

# DATASET PIPELINE

In [9]:
def triple_mask(img, mask):
    return img, (mask, mask, mask)

train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_masks))
train_ds = train_ds.shuffle(len(train_images), seed=SEED)
train_ds = train_ds.map(parse_sample, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.map(augment, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.map(triple_mask)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_images, val_masks))
val_ds = val_ds.map(parse_sample)
val_ds = val_ds.map(triple_mask)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# BUILD MODEL (UNET WITH DEEP SUPERVISION)

In [10]:
inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 1))

c1 = tf.keras.layers.Conv2D(64,3,padding="same",activation="relu")(inputs)
c1 = tf.keras.layers.Conv2D(64,3,padding="same",activation="relu")(c1)
p1 = tf.keras.layers.MaxPooling2D()(c1)

c2 = tf.keras.layers.Conv2D(128,3,padding="same",activation="relu")(p1)
c2 = tf.keras.layers.Conv2D(128,3,padding="same",activation="relu")(c2)
p2 = tf.keras.layers.MaxPooling2D()(c2)

c3 = tf.keras.layers.Conv2D(256,3,padding="same",activation="relu")(p2)
c3 = tf.keras.layers.Conv2D(256,3,padding="same",activation="relu")(c3)
p3 = tf.keras.layers.MaxPooling2D()(c3)

c4 = tf.keras.layers.Conv2D(512,3,padding="same",activation="relu")(p3)
c4 = tf.keras.layers.Conv2D(512,3,padding="same",activation="relu")(c4)
p4 = tf.keras.layers.MaxPooling2D()(c4)

b = tf.keras.layers.Conv2D(1024,3,padding="same",activation="relu")(p4)
b = tf.keras.layers.Conv2D(1024,3,padding="same",activation="relu")(b)

u1 = tf.keras.layers.Conv2DTranspose(512,2,strides=2,padding="same")(b)
u1 = tf.keras.layers.Concatenate()([u1,c4])
u1 = tf.keras.layers.Conv2D(512,3,padding="same",activation="relu")(u1)

u2 = tf.keras.layers.Conv2DTranspose(256,2,strides=2,padding="same")(u1)
u2 = tf.keras.layers.Concatenate()([u2,c3])
u2 = tf.keras.layers.Conv2D(256,3,padding="same",activation="relu")(u2)

d2_out = tf.keras.layers.Conv2D(1,1,activation="sigmoid",name="d2_out")(tf.keras.layers.UpSampling2D(4)(u2))

u3 = tf.keras.layers.Conv2DTranspose(128,2,strides=2,padding="same")(u2)
u3 = tf.keras.layers.Concatenate()([u3,c2])
u3 = tf.keras.layers.Conv2D(128,3,padding="same",activation="relu")(u3)

d3_out = tf.keras.layers.Conv2D(1,1,activation="sigmoid",name="d3_out")(tf.keras.layers.UpSampling2D(2)(u3))

u4 = tf.keras.layers.Conv2DTranspose(64,2,strides=2,padding="same")(u3)
u4 = tf.keras.layers.Concatenate()([u4,c1])
u4 = tf.keras.layers.Conv2D(64,3,padding="same",activation="relu")(u4)

final_out = tf.keras.layers.Conv2D(1,1,activation="sigmoid",name="final_out")(u4)

model = tf.keras.Model(inputs, [d2_out, d3_out, final_out])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 384, 384,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 384, 384,  │        640 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 384, 384,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 192, 192,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 192, 192,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 192, 192,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 96, 96,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 96, 96,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 96, 96,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 48, 48,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 48, 48,    │  1,180,160 │ max_pooling2d_2[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 48, 48,    │  2,359,808 │ conv2d_6[0][0]    │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 24, 24,    │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 24, 24,    │  4,719,616 │ max_pooling2d_3[… │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 24, 24,    │  9,438,208 │ conv2d_8[0][0]    │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 48, 48,    │  2,097,664 │ conv2d_9[0][0]    │
│ (Conv2DTranspose)   │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 48, 48,    │          0 │ conv2d_transpose

 Total params: 27,896,579 (106.42 MB)

 Trainable params: 27,896,579 (106.42 MB)

 Non-trainable params: 0 (0.00 B)

# LOSS AND METRICS

In [11]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true*y_pred)

    return (2*intersection + smooth) / (tf.reduce_sum(y_true)+tf.reduce_sum(y_pred)+smooth)


def dice_loss(y_true,y_pred):

    return 1 - dice_coefficient(y_true,y_pred)


bce = tf.keras.losses.BinaryCrossentropy()


def combined_loss(y_true,y_pred):

    return dice_loss(y_true,y_pred) + bce(y_true,y_pred)

# COSINE LEARNING RATE

In [12]:
steps = (len(train_images)//BATCH_SIZE) * EPOCHS

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=steps
)

optimizer = tf.keras.optimizers.AdamW(lr_schedule)

# CALLBACKS

In [13]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.h5",
        monitor="val_final_out_dice_coefficient",
        mode="max",
        save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_final_out_dice_coefficient",
        patience=8,
        restore_best_weights=True
    )
]

# PHASE 1 TRAINING

In [14]:
model.compile(
    optimizer=optimizer,
    loss={"d2_out":combined_loss,"d3_out":combined_loss,"final_out":combined_loss},
    loss_weights={"d2_out":0.2,"d3_out":0.3,"final_out":0.5},
    metrics={"final_out":[dice_coefficient]}
)

print("Phase-1 Training")

hist_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Phase-1 Training
Epoch 1/80
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 778ms/step - d2_out_loss: nan - d3_out_loss: nan - final_out_dice_coefficient: nan - final_out_loss: nan - loss: nan

KeyboardInterrupt: 

# PHASE 2 FINE TUNING

In [ ]:
best_dice = max(hist_phase1.history["val_final_out_dice_coefficient"])
print("Best Dice:", best_dice)

if best_dice < TARGET_DICE:

    print("Running Fine-Tune Phase")

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(1e-5),
        loss={"d2_out":combined_loss,"d3_out":combined_loss,"final_out":combined_loss},
        loss_weights={"d2_out":0.2,"d3_out":0.3,"final_out":0.5},
        metrics={"final_out":[dice_coefficient]}
    )

    hist_phase2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=25,
        callbacks=callbacks
    )

# GRAPH GENERATION

In [ ]:
plt.plot(hist_phase1.history["final_out_dice_coefficient"])
plt.plot(hist_phase1.history["val_final_out_dice_coefficient"])

plt.title("Dice Score")
plt.legend(["train","val"])

plt.show()

# TEST TIME AUGMENTATION

In [ ]:
def predict_tta(model, batch):

    p1 = model.predict(batch, verbose=0)[2]

    p2 = model.predict(tf.image.flip_left_right(batch), verbose=0)[2]
    p2 = tf.image.flip_left_right(p2)

    return (p1 + p2)/2

# EVALUATION

In [ ]:
test_ds = tf.data.Dataset.from_tensor_slices((test_images,test_masks))

test_ds = test_ds.map(parse_sample)
test_ds = test_ds.map(triple_mask)
test_ds = test_ds.batch(BATCH_SIZE)

print("Evaluating...")

model.evaluate(test_ds)

# SEGMENTATION METRICS (CONFUSION MATRIX + SCORES)

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

y_true_pixels = []
y_pred_pixels = []

for imgs, (masks, _, _) in test_ds:

    preds = model.predict(imgs, verbose=0)[2]
    preds_binary = (preds > 0.5).astype(np.uint8)

    masks_np = masks.numpy().astype(np.uint8)

    y_true_pixels.extend(masks_np.flatten())
    y_pred_pixels.extend(preds_binary.flatten())

y_true_pixels = np.array(y_true_pixels)
y_pred_pixels = np.array(y_pred_pixels)

cm = confusion_matrix(y_true_pixels, y_pred_pixels)

TN, FP, FN, TP = cm.ravel()

precision = TP / (TP + FP + 1e-8)
recall = TP / (TP + FN + 1e-8)
iou = TP / (TP + FP + FN + 1e-8)
dice = (2 * TP) / (2 * TP + FP + FN + 1e-8)
pixel_accuracy = (TP + TN) / (TP + TN + FP + FN + 1e-8)

print("\n=== SEGMENTATION METRICS ===")
print("Dice Score:", dice)
print("IoU Score:", iou)
print("Precision:", precision)
print("Recall:", recall)
print("Pixel Accuracy:", pixel_accuracy)

plt.figure(figsize=(5,5))
plt.imshow(cm, cmap="Blues")
plt.title("Pixel Confusion Matrix")
plt.colorbar()

plt.xticks([0,1], ["Background","Tumor"])
plt.yticks([0,1], ["Background","Tumor"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha="center", va="center", color="black")

plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# SAVE MODEL

In [ ]:
model.save(MODEL_KERAS)
model.save(MODEL_H5)

print("Saved models:")
print(MODEL_KERAS)
print(MODEL_H5)